# LoRa Predictive and Optimization Model

### Configuration and Data Variables

In [ ]:
# Load environment variables
load_dotenv()
warnings.filterwarnings('ignore')

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('lora_system.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

@dataclass
class LoRaParameters:
    """LoRa communication parameters"""
    tx_power: float = 14.0  # dBm
    spreading_factor: int = 7
    frequency: float = 868.0  # MHz
    bandwidth: float = 125.0  # kHz
    coding_rate: int = 4  # 4/5

@dataclass
class PathPoint:
    """Represents a point in the path with all attributes"""
    lat: float
    lon: float
    elevation: float = 0.0
    land_cover: int = 50
    terrain_penalty: float = 0.3
    rssi: float = -100.0
    snr: float = 0.0
    pdr: float = 0.5
    path_loss: float = 100.0
    distance_to_start: float = 0.0
    distance_to_destination: float = 0.0
    grid_x: int = 0
    grid_y: int = 0
    is_relay: bool = False
    hop_number: int = 0

@dataclass
class OptimizationConfig:
    """Configuration for path optimization"""
    grid_spacing_km: float = 1.5
    max_hop_distance_km: float = 5.0
    min_relay_distance_km: float = 1.0
    interpolate_between_points: bool = True
    use_real_data: bool = True
    adaptive_grid: bool = True
    path_smoothing: bool = True


### 1. Google Earth Engine Integration

In [ ]:
class GoogleEarthEngineIntegration:
    """Robust real-time spatial data fetching from Google Earth Engine"""
    
    def __init__(self, use_service_account=False, service_account_key=None):
        self.initialized = False
        self.cache = {}
        self.rate_limiter = RateLimiter(calls_per_second=10)
        self.use_fallback = True  # Force fallback mode by default
        
        try:
            # Try to initialize with explicit error handling
            try:
                # First, try to initialize with a project ID if available
                project_id = os.getenv('GEE_PROJECT_ID')
                if project_id:
                    ee.Initialize(project=project_id)
                    logger.info(f"Google Earth Engine initialized with project: {project_id}")
                else:
                    # Try without project ID
                    ee.Initialize()
                    logger.info("Google Earth Engine initialized without project ID")
                
                # Test initialization with a simple request
                test_point = ee.Geometry.Point([0, 0])
                test_result = ee.Image('USGS/SRTMGL1_003').sample(test_point, scale=30).getInfo()
                
                if test_result and len(test_result['features']) > 0:
                    logger.info("Google Earth Engine test successful!")
                    self.initialized = True
                    self.use_fallback = False  # Only use GEE if it's working
                else:
                    logger.warning("Google Earth Engine test returned empty results")
                    self.initialized = False
                    
            except Exception as init_error:
                logger.warning(f"Google Earth Engine initialization failed: {init_error}")
                self.initialized = False
        
        except Exception as e:
            logger.warning(f"Could not initialize Google Earth Engine: {e}")
            self.initialized = False
        
        # Land cover penalty mapping
        self.land_cover_penalties = {
            10: 0.4, 20: 0.3, 30: 0.2, 40: 0.25, 50: 0.4,
            60: 0.15, 70: 0.05, 80: 0.0, 90: 0.35, 95: 0.3, 100: 0.1
        }
    
    def get_elevation(self, lat: float, lon: float, retry_count=3) -> float:
        """Fetch real elevation data from SRTM (30m resolution) with retry logic"""
        if not self.initialized or self.use_fallback:
            return self._simulate_elevation(lat, lon)
        
        cache_key = f"elevation_{lat:.6f}_{lon:.6f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        for attempt in range(retry_count):
            try:
                with self.rate_limiter:
                    point = ee.Geometry.Point([lon, lat])
                    srtm = ee.Image('USGS/SRTMGL1_003')
                    
                    elevation_dict = srtm.reduceRegion(
                        reducer=ee.Reducer.first(),
                        geometry=point,
                        scale=30,
                        maxPixels=1
                    ).getInfo()
                    
                    elevation = elevation_dict.get('elevation')
                    
                    if elevation is not None:
                        elevation = float(elevation)
                        self.cache[cache_key] = elevation
                        return elevation
                    else:
                        return self._simulate_elevation(lat, lon)
                        
            except Exception as e:
                if attempt == retry_count - 1:
                    logger.warning(f"Failed to get elevation for ({lat}, {lon}): {e}")
                    return self._simulate_elevation(lat, lon)
                time.sleep(0.5)
        
        return self._simulate_elevation(lat, lon)
    
    def get_land_cover(self, lat: float, lon: float, retry_count=3) -> Tuple[int, float]:
        """Fetch real land cover data from ESA WorldCover (10m resolution)"""
        if not self.initialized or self.use_fallback:
            return self._simulate_land_cover(lat, lon)
        
        cache_key = f"landcover_{lat:.6f}_{lon:.6f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        for attempt in range(retry_count):
            try:
                with self.rate_limiter:
                    point = ee.Geometry.Point([lon, lat])
                    worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
                    
                    lc_dict = worldcover.reduceRegion(
                        reducer=ee.Reducer.first(),
                        geometry=point,
                        scale=10,
                        maxPixels=1
                    ).getInfo()
                    
                    land_cover = lc_dict.get('Map')
                    
                    if land_cover is not None:
                        land_cover_code = int(land_cover)
                        terrain_penalty = self.land_cover_penalties.get(land_cover_code, 0.5)
                        result = (land_cover_code, terrain_penalty)
                        self.cache[cache_key] = result
                        return result
                    else:
                        return 80, 0.0  # Water body with no penalty
                        
            except Exception as e:
                if attempt == retry_count - 1:
                    logger.warning(f"Failed to get land cover for ({lat}, {lon}): {e}")
                    return self._simulate_land_cover(lat, lon)
                time.sleep(0.5)
        
        return self._simulate_land_cover(lat, lon)
    
    def get_spatial_features(self, lat: float, lon: float) -> Dict:
        """Fetch all spatial features for a location with caching"""
        cache_key = f"spatial_{lat:.6f}_{lon:.6f}"
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        elevation = self.get_elevation(lat, lon)
        land_cover, terrain_penalty = self.get_land_cover(lat, lon)
        
        result = {
            'elevation': elevation,
            'land_cover': land_cover,
            'terrain_penalty': terrain_penalty
        }
        self.cache[cache_key] = result
        return result
    
    def batch_get_spatial_features(self, coordinates: List[Tuple[float, float]], 
                                   batch_size: int = 50,
                                   use_batch_mode: bool = True) -> List[Dict]:
        """Fetch spatial features for multiple locations with optimized batching"""
        total = len(coordinates)
        logger.info(f"Fetching spatial features for {total} locations...")
        
        if use_batch_mode and self.initialized and not self.use_fallback:
            return self._batch_get_spatial_features_efficient(coordinates)
        else:
            logger.info("Using sequential mode (GEE not available or disabled)")
            return self._batch_get_spatial_features_sequential(coordinates, batch_size)
    
    def _batch_get_spatial_features_efficient(self, coordinates: List[Tuple[float, float]]) -> List[Dict]:
        """Efficient batch fetching using GEE's batch capabilities"""
        results = []
        errors = 0
        successes = 0
        
        try:
            # Create points as a FeatureCollection
            features = []
            for i, (lat, lon) in enumerate(coordinates):
                point = ee.Geometry.Point([lon, lat])
                features.append(ee.Feature(point, {'index': i}))
            
            points_fc = ee.FeatureCollection(features)
            
            # Sample elevation
            srtm = ee.Image('USGS/SRTMGL1_003')
            elevation_sampled = srtm.reduceRegions(
                collection=points_fc,
                reducer=ee.Reducer.first(),
                scale=30
            )
            
            # Sample land cover
            worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
            landcover_sampled = worldcover.reduceRegions(
                collection=elevation_sampled,
                reducer=ee.Reducer.first(),
                scale=10
            )
            
            # Get results
            sampled_data = landcover_sampled.getInfo()
            
            # Process results
            result_dict = {}
            for feature in sampled_data['features']:
                idx = feature['properties']['index']
                elevation = feature['properties'].get('elevation', None)
                land_cover = feature['properties'].get('Map', None)
                
                if elevation is None:
                    elevation = self._simulate_elevation(coordinates[idx][0], coordinates[idx][1])
                else:
                    elevation = float(elevation)
                    successes += 1
                
                if land_cover is None:
                    land_cover = 50
                    terrain_penalty = 0.3
                    errors += 1
                else:
                    land_cover = int(land_cover)
                    terrain_penalty = self.land_cover_penalties.get(land_cover, 0.5)
                    successes += 1
                
                result_dict[idx] = {
                    'elevation': elevation,
                    'land_cover': land_cover,
                    'terrain_penalty': terrain_penalty,
                    'latitude': coordinates[idx][0],
                    'longitude': coordinates[idx][1]
                }
            
            # Ensure all coordinates have results in order
            for i in range(len(coordinates)):
                if i in result_dict:
                    results.append(result_dict[i])
                else:
                    # Missing data, use simulation
                    lat, lon = coordinates[i]
                    results.append({
                        'elevation': self._simulate_elevation(lat, lon),
                        'land_cover': 50,
                        'terrain_penalty': 0.3,
                        'latitude': lat,
                        'longitude': lon
                    })
                    errors += 1
            
            logger.info(f"Batch fetch completed - Successes: {successes}, Errors/Missing: {errors}")
            
        except Exception as e:
            logger.warning(f"Batch mode failed ({e}), falling back to sequential mode...")
            return self._batch_get_spatial_features_sequential(coordinates, 50)
        
        return results
    
    def _batch_get_spatial_features_sequential(self, coordinates: List[Tuple[float, float]], 
                                               batch_size: int = 50) -> List[Dict]:
        """Sequential fetching with progress tracking and rate limiting"""
        results = []
        total = len(coordinates)
        errors = 0
        successes = 0
        
        for i, (lat, lon) in enumerate(coordinates):
            if i > 0 and i % batch_size == 0:
                logger.info(f"Progress: {i}/{total} ({i/total*100:.1f}%) - Successes: {successes}, Errors: {errors}")
                time.sleep(1)  # Rate limiting
            
            try:
                features = self.get_spatial_features(lat, lon)
                features['latitude'] = lat
                features['longitude'] = lon
                results.append(features)
                successes += 1
            except Exception as e:
                # If total failure, use simulation
                features = {
                    'elevation': self._simulate_elevation(lat, lon),
                    'land_cover': 50,
                    'terrain_penalty': 0.3,
                    'latitude': lat,
                    'longitude': lon
                }
                results.append(features)
                errors += 1
        
        logger.info(f"Completed: {total}/{total} (100%) - Successes: {successes}, Errors: {errors}")
        
        if errors > 0:
            logger.warning(f"{errors} locations used simulated data (GEE data unavailable)")
        
        return results
    
    def _simulate_elevation(self, lat: float, lon: float) -> float:
        """Simulate elevation when GEE is not available"""
        return abs(np.sin(lat * 10) * np.cos(lon * 10) * 500)
    
    def _simulate_land_cover(self, lat: float, lon: float) -> Tuple[int, float]:
        """Simulate land cover when GEE is not available"""
        land_cover_codes = [10, 20, 30, 40, 50, 60, 70, 80, 90, 95, 100]
        land_cover = np.random.choice(land_cover_codes)
        terrain_penalty = self.land_cover_penalties.get(land_cover, 0.5)
        return land_cover, terrain_penalty

class RateLimiter:
    """Simple rate limiter for API calls"""
    def __init__(self, calls_per_second=10):
        self.calls_per_second = calls_per_second
        self.last_call = 0
        
    def __enter__(self):
        elapsed = time.time() - self.last_call
        if elapsed < 1.0 / self.calls_per_second:
            time.sleep((1.0 / self.calls_per_second) - elapsed)
        self.last_call = time.time()
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        pass


### 2. Data Loading and Preprocessing

In [ ]:
class LoRaDataPreprocessor:
    """Robust data loading and preprocessing with flexible format handling"""
    
    def __init__(self, gee_integration: Optional[GoogleEarthEngineIntegration] = None):
        self.scaler = StandardScaler()
        self.gee = gee_integration
        
    def load_dataset1(self, filepath):
        """Load dataset with format: latitude,longitude,elevation,land_cover,etc."""
        try:
            # Try different separators
            for sep in [',', '|', '\t']:
                try:
                    df = pd.read_csv(filepath, sep=sep)
                    if len(df.columns) > 5:  # Reasonable number of columns
                        break
                except:
                    continue
            else:
                raise ValueError("Could not determine file format")
            
            # Map column names flexibly
            column_mapping = {
                'latitude': ['latitude', 'lat'],
                'longitude': ['longitude', 'lon'],
                'elevation': ['elevation', 'altitude', 'elev'],
                'land_cover': ['land_cover', 'land_cover_code', 'landcover'],
                'terrain_penalty': ['terrain_penalty', 'terrain'],
                'RSSI': ['RSSI', 'rssi'],
                'SNR': ['SNR', 'snr'],
                'PDR': ['PDR', 'pdr'],
                'observed_path_loss': ['observed_path_loss', 'path_loss', 'loss'],
                'spreading_factor': ['spreading_factor', 'sf'],
                'frequency': ['frequency', 'freq'],
                'tx_power': ['tx_power', 'power'],
                'distance_to_start': ['distance_to_start'],
                'distance_to_destination': ['distance_to_destination']
            }
            
            # Create standardized dataframe
            df_processed = pd.DataFrame()
            for std_col, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns:
                        df_processed[std_col] = df[col]
                        break
                else:
                    # Set default values if column not found
                    if std_col == 'spreading_factor':
                        df_processed[std_col] = 7
                    elif std_col == 'frequency':
                        df_processed[std_col] = 868
                    elif std_col == 'tx_power':
                        df_processed[std_col] = 14
                    elif std_col in ['terrain_penalty']:
                        df_processed[std_col] = 0.3
                    else:
                        df_processed[std_col] = 0
            
            return df_processed.dropna(subset=['RSSI', 'SNR', 'PDR'])
        
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            return pd.DataFrame()
    
    def load_dataset2(self, filepath):
        """Load dataset with format: device_id,gateway_id,latitude,longitude,altitude,etc."""
        try:
            # Try different separators
            for sep in [',', '|', '\t']:
                try:
                    df = pd.read_csv(filepath, sep=sep)
                    if len(df.columns) > 5:  # Reasonable number of columns
                        break
                except:
                    continue
            else:
                raise ValueError("Could not determine file format")
            
            # Map column names flexibly
            column_mapping = {
                'latitude': ['latitude', 'lat'],
                'longitude': ['longitude', 'lon'],
                'elevation': ['altitude', 'elevation', 'elev'],
                'land_cover': ['land_cover_code', 'land_cover', 'landcover'],
                'terrain_penalty': ['terrain_penalty', 'terrain'],
                'RSSI': ['RSSI', 'rssi'],
                'SNR': ['SNR', 'snr'],
                'PDR': ['PDR', 'pdr'],
                'observed_path_loss': ['observed_path_loss', 'path_loss', 'loss'],
                'spreading_factor': ['spreading_factor', 'sf'],
                'frequency': ['frequency', 'freq'],
                'tx_power': ['tx_power', 'power'],
                'distance_to_start': ['distance_to_start'],
                'distance_to_destination': ['distance_to_destination']
            }
            
            # Create standardized dataframe
            df_processed = pd.DataFrame()
            for std_col, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns:
                        df_processed[std_col] = df[col]
                        break
                else:
                    # Set default values if column not found
                    if std_col == 'spreading_factor':
                        df_processed[std_col] = 7
                    elif std_col == 'frequency':
                        df_processed[std_col] = 868
                    elif std_col == 'tx_power':
                        df_processed[std_col] = 14
                    elif std_col in ['terrain_penalty']:
                        df_processed[std_col] = 0.3
                    else:
                        df_processed[std_col] = 0
            
            return df_processed.dropna(subset=['RSSI', 'SNR', 'PDR'])
        
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            return pd.DataFrame()
    
    def merge_datasets(self, df1, df2):
        """Merge and clean datasets with validation"""
        if df1.empty and df2.empty:
            raise ValueError("Both datasets are empty!")
        
        if df1.empty:
            df_combined = df2.copy()
        elif df2.empty:
            df_combined = df1.copy()
        else:
            df_combined = pd.concat([df1, df2], ignore_index=True)
        
        # Drop rows with critical missing values
        df_combined = df_combined.dropna(subset=['RSSI', 'SNR', 'PDR'])
        
        # Feature engineering
        df_combined['distance_total'] = df_combined['distance_to_start'] + df_combined['distance_to_destination']
        df_combined['elevation_normalized'] = df_combined['elevation'] / 1000
        df_combined['rssi_normalized'] = (df_combined['RSSI'] + 130) / 100
        
        # Add default values if columns don't exist
        if 'frequency' not in df_combined.columns:
            df_combined['frequency'] = 868  # Default EU868
            logger.warning("⚠️  'frequency' column not found, using default 868 MHz")
        
        if 'tx_power' not in df_combined.columns:
            df_combined['tx_power'] = 14  # Default 14 dBm
            logger.warning("⚠️  'tx_power' column not found, using default 14 dBm")
        
        logger.info(f"Combined dataset shape: {df_combined.shape}")
        logger.info(f"Missing values:\n{df_combined.isnull().sum()}")
        
        return df_combined
    
    def prepare_features(self, df, target_cols=['RSSI', 'SNR', 'PDR', 'observed_path_loss']):
        """Prepare feature and target matrices with validation"""
        feature_cols = [
            'elevation', 'land_cover', 'terrain_penalty',
            'distance_to_start', 'distance_to_destination',
            'spreading_factor', 'frequency', 'tx_power',
            'distance_total', 'elevation_normalized'
        ]
        
        # Validate all feature columns exist
        missing_cols = [col for col in feature_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing feature columns: {missing_cols}")
        
        X = df[feature_cols].values
        y = df[target_cols].values
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        logger.info(f"Training samples: {len(X_train)}")
        logger.info(f"Test samples: {len(X_test)}")
        logger.info(f"Features: {len(feature_cols)}")
        
        return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols


### 3. Pytorch Neural Network Model

In [ ]:
class LoRaDataset(Dataset):
    """PyTorch dataset for LoRa data"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LoRaNeuralNetwork(nn.Module):
    """Advanced neural network with configurable architecture"""
    def __init__(self, input_size, output_size, config=None):
        super(LoRaNeuralNetwork, self).__init__()
        
        # Default configuration
        default_config = {
            'hidden_sizes': [256, 128, 64, 32],
            'dropout_rate': 0.3,
            'activation': 'relu',
            'batch_norm': True,
            'residual_connections': True
        }
        
        if config:
            default_config.update(config)
        
        self.config = default_config
        layers = []
        prev_size = input_size
        
        for i, hidden_size in enumerate(self.config['hidden_sizes']):
            layers.append(nn.Linear(prev_size, hidden_size))
            
            if self.config['batch_norm']:
                layers.append(nn.BatchNorm1d(hidden_size))
            
            if self.config['activation'] == 'relu':
                layers.append(nn.ReLU())
            elif self.config['activation'] == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.2))
            elif self.config['activation'] == 'elu':
                layers.append(nn.ELU())
            
            if self.config['dropout_rate'] > 0:
                layers.append(nn.Dropout(self.config['dropout_rate']))
            
            # Add residual connection for deeper networks
            if self.config['residual_connections'] and i > 0 and prev_size == hidden_size:
                layers.append(ResidualConnection())
            
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        self.network = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.network(x)

    def predict(self, X):
        """Make predictions with the model"""
        self.eval()
        with torch.no_grad():
            if isinstance(X, np.ndarray):
                X = torch.FloatTensor(X)
            X = X.to(next(self.parameters()).device)
            predictions = self.forward(X).cpu().numpy()
        return predictions

class ResidualConnection(nn.Module):
    """Residual connection for deeper networks"""
    def __init__(self):
        super(ResidualConnection, self).__init__()
        
    def forward(self, x):
        return x

class NeuralNetworkTrainer:
    """Advanced neural network trainer with early stopping and learning rate scheduling"""
    def __init__(self, input_size, output_size, device, config=None):
        self.device = device
        self.config = config or {}
        
        # Model configuration
        model_config = self.config.get('model', {})
        self.model = LoRaNeuralNetwork(input_size, output_size, model_config).to(device)
        
        # Training configuration
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(
            self.model.parameters(), 
            lr=self.config.get('learning_rate', 0.001),
            weight_decay=self.config.get('weight_decay', 1e-5)
        )
        
        # Learning rate scheduling
        scheduler_type = self.config.get('scheduler', 'reduce_on_plateau')
        if scheduler_type == 'reduce_on_plateau':
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer, mode='min', patience=10, factor=0.5, verbose=True
            )
        elif scheduler_type == 'cosine':
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=self.config.get('epochs', 100)
            )
        
        # Early stopping
        self.early_stopping_patience = self.config.get('early_stopping_patience', 20)
        self.early_stopping_counter = 0
        self.best_val_loss = float('inf')
        
        self.train_losses = []
        self.val_losses = []
        
    def train(self, train_loader, val_loader, epochs=None):
        """Train the neural network with early stopping"""
        epochs = epochs or self.config.get('epochs', 100)
        logger.info(f"\nTraining Neural Network on {self.device}...")
        logger.info(f"Model architecture: {self.model.config}")
        logger.info(f"Training configuration: {self.config}")
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_steps = 0
            
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = self.criterion(outputs, y_batch)
                loss.backward()
                
                # Gradient clipping
                if self.config.get('gradient_clip', 0) > 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
                
                self.optimizer.step()
                
                train_loss += loss.item()
                train_steps += 1
            
            train_loss /= train_steps
            self.train_losses.append(train_loss)
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_steps = 0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                    outputs = self.model(X_batch)
                    loss = self.criterion(outputs, y_batch)
                    val_loss += loss.item()
                    val_steps += 1
            
            val_loss /= val_steps
            self.val_losses.append(val_loss)
            
            # Learning rate scheduling
            if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_loss)
            else:
                self.scheduler.step()
            
            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.early_stopping_counter = 0
                # Save best model
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                self.early_stopping_counter += 1
                if self.early_stopping_counter >= self.early_stopping_patience:
                    logger.info(f"Early stopping at epoch {epoch+1}")
                    # Load best model
                    self.model.load_state_dict(torch.load('best_model.pth'))
                    break
            
            if (epoch + 1) % 10 == 0:
                logger.info(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        logger.info("Neural Network training completed!")
        
    def predict(self, X):
        """Make predictions with the trained model"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.model(X_tensor).cpu().numpy()
        return predictions


### 4. Random Forest Model

### 5. XGBoost Model

### 6. Path Optimization with A* Algorithm

### 7. MultiHop Route Planning

### 8. Coverage Map Generator

### 9. Visulization and Evaluation

### 10. Main Execution Pipeline